In [0]:
# Databricks notebook
# Purpose: Extract the latest SUSEP ZIP file from Bronze ZIP folder into Bronze CSV folder

import os
import zipfile
import shutil

# ============================================================
# 1. Configuration
# ============================================================

base_zip_path = "/Volumes/workspace/susep_bronze/lakehouse/lakehouse-susep/bronze/raw/zip"
base_csv_path = "/Volumes/workspace/susep_bronze/lakehouse/lakehouse-susep/bronze/raw/csv"

expected_zip_file_name = "BaseCompleta.zip"

print(f"ZIP base path: {base_zip_path}")
print(f"CSV base path: {base_csv_path}")

In [0]:
# ============================================================
# 2. Function adjust path name
# ============================================================

def to_posix_path(path: str) -> str:
    """
    Converts Databricks dbfs:/ paths to POSIX paths usable by Python libraries.
    
    Example:
    dbfs:/Volumes/catalog/schema/volume/file.zip
    becomes:
    /Volumes/catalog/schema/volume/file.zip
    """
    if path.startswith("dbfs:/Volumes/"):
        return path.replace("dbfs:/Volumes/", "/Volumes/", 1)
    
    if path.startswith("dbfs:/"):
        return path.replace("dbfs:/", "/dbfs/", 1)
    
    return path

In [0]:
# ============================================================
# 3. Identify latest ZIP folder
# ============================================================

zip_folders = [
    item for item in dbutils.fs.ls(base_zip_path)
    if item.isDir()
]

if not zip_folders:
    raise FileNotFoundError(f"No folders found in ZIP directory: {base_zip_path}")

latest_zip_folder = sorted(zip_folders, key=lambda x: x.name, reverse=True)[0]

latest_folder_name = latest_zip_folder.name.replace("/", "")
latest_zip_folder_path = to_posix_path(latest_zip_folder.path.rstrip("/"))

print(f"Latest folder name: {latest_folder_name}")
print(f"Latest ZIP folder path: {latest_zip_folder_path}")

In [0]:
# ============================================================
# 4. Locate ZIP file inside latest folder
# ============================================================

# For dbutils.fs.ls we can use the normal Volume path
latest_zip_folder_dbutils_path = f"{base_zip_path}/{latest_folder_name}"

files_in_latest_folder = dbutils.fs.ls(latest_zip_folder_dbutils_path)

zip_files = [
    item for item in files_in_latest_folder
    if item.name.lower().endswith(".zip")
]

if not zip_files:
    raise FileNotFoundError(f"No ZIP file found in folder: {latest_zip_folder_dbutils_path}")

matching_zip_files = [
    item for item in zip_files
    if item.name == expected_zip_file_name
]

if matching_zip_files:
    zip_file = matching_zip_files[0]
else:
    zip_file = zip_files[0]

zip_file_path = to_posix_path(zip_file.path)

print(f"ZIP file found: {zip_file_path}")
print(f"ZIP file size: {zip_file.size:,} bytes")

In [0]:
# ============================================================
# 5. Create matching CSV target folder
# ============================================================

target_csv_folder = f"{base_csv_path}/{latest_folder_name}"

dbutils.fs.mkdirs(target_csv_folder)

print(f"Target CSV folder created or already exists: {target_csv_folder}")

In [0]:
# ============================================================
# 6. Extract CSV files from ZIP into target folder
# ============================================================

extracted_files = []

with zipfile.ZipFile(zip_file_path, "r") as zip_ref:
    
    zip_members = zip_ref.namelist()
    
    csv_members = [
        member for member in zip_members
        if member.lower().endswith(".csv")
    ]
    
    if not csv_members:
        raise FileNotFoundError("No CSV files found inside the ZIP file.")
    
    print(f"CSV files found inside ZIP: {len(csv_members)}")
    
    for member in csv_members:
        output_file_name = os.path.basename(member)
        
        if not output_file_name:
            continue
        
        output_file_path = f"{target_csv_folder}/{output_file_name}"
        
        with zip_ref.open(member) as source_file:
            with open(output_file_path, "wb") as target_file:
                shutil.copyfileobj(source_file, target_file)
        
        extracted_files.append(output_file_path)
        print(f"Extracted: {output_file_path}")

print("\nExtraction completed successfully.")
print(f"Total CSV files extracted: {len(extracted_files)}")